# 03 — Emotion Labeling & User-Level Aggregation

Labels each post with 27 GoEmotions categories, then aggregates to user level using max pooling.

**Input:** `reddit_estrangement_labeled_final.csv`, `reddit_estrangement_clean.csv`  
**Output:** `reddit_estrangement_pooled_emotions.csv`

In [ ]:
!pip install transformers torch tqdm --quiet

In [ ]:
# Load data
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/Raw Data/reddit_estrangement_labeled_final.csv")
df_clean = pd.read_csv("/content/drive/MyDrive/Raw Data/reddit_estrangement_clean.csv")
df = df.merge(df_clean, on="post_id", how="inner")
print(f"Loaded: {len(df)} posts | {len(df.columns)} columns")

In [ ]:
# User-level aggregation — OR pooling for causal attributions
label_cols = [
    'abuse', 'family_conflict', 'deceit', 'divorce_remarriage',
    'substance_use', 'ingratitude_entitlement', 'toxic_cruelty',
    'toxic_disrespect', 'toxic_anger', 'toxic_boundary_violations',
    'toxic_generic', 'voluntary_distance', 'involuntary_distance',
    'parents_partner', 'childs_partner', 'mental_health',
    'selfishness', 'disapproval', 'exclusion', 'emotional_neglect',
    'material_deprivation', 'unknown_reason'
]

# User-level aggregation: OR pooling for causal attributions
# A user is marked 1 for a category if it appeared in ANY of their posts
df_pooled = df.groupby("author_anon").agg(
    post_count=("post_id", "count"),
    post_ids=("post_id", lambda x: "|".join(x.astype(str))),
    subreddit=("subreddit", "first"),
    date_first=("date", "min"),
    date_last=("date", "max"),
    **{col: (col, "max") for col in label_cols}
).reset_index()

print(f"Users: {len(df_pooled)}")
print(f"Posts per user — mean: {df_pooled['post_count'].mean():.2f}, median: {df_pooled['post_count'].median()}")

In [ ]:
# Load GoEmotions model
from transformers import pipeline
from tqdm import tqdm

print("Loading GoEmotions model...")
emotion_classifier = pipeline(
    task="text-classification",
    model="SamLowe/roberta-base-go_emotions",
    top_k=None,
    device=0,
    truncation=True,
    max_length=512,
)
print("Model ready ✓")

In [ ]:
# Emotion labeling — GoEmotions
BATCH_SIZE = 32
THRESHOLD  = 0.3

texts    = df["text"].tolist()
post_ids = df["post_id"].tolist()
all_results = []

print(f"Labeling {len(texts)} posts...")

for i in tqdm(range(0, len(texts), BATCH_SIZE)):
    batch_texts = texts[i:i + BATCH_SIZE]
    batch_ids   = post_ids[i:i + BATCH_SIZE]
    outputs     = emotion_classifier(batch_texts)

    for post_id, output in zip(batch_ids, outputs):
        scores   = {item["label"]: item["score"] for item in output}
        active   = [l for l, s in scores.items() if s >= THRESHOLD]
        dominant = max(scores, key=scores.get)

        row = {
            "post_id"         : post_id,
            "dominant_emotion": dominant,
            "active_emotions" : "|".join(active) if active else "neutral"
        }
        for emotion, score in scores.items():
            row[f"score_{emotion}"] = round(score, 3)

        all_results.append(row)

df_emotions = pd.DataFrame(all_results)
print(f"Labeling complete ✓ — {len(df_emotions)} posts")

In [ ]:
# Merge emotion scores with post data
df_with_emotions = df.merge(df_emotions, on="post_id", how="inner")
print(f"Merged: {len(df_with_emotions)} posts")

In [ ]:
# User-level aggregation — max pooling for emotions
import collections

ALL_EMOTIONS = [
    "admiration", "amusement", "anger", "annoyance", "approval",
    "caring", "confusion", "curiosity", "disappointment", "disapproval",
    "disgust", "embarrassment", "excitement", "fear", "gratitude",
    "grief", "joy", "love", "nervousness", "optimism", "pride",
    "realization", "relief", "remorse", "sadness", "surprise", "neutral"
]

THRESHOLD = 0.3

def aggregate_emotions(group):
    scores = {}
    for e in ALL_EMOTIONS:
        col = f"score_{e}"
        if col in group.columns:
            scores[f"max_{e}"] = round(group[col].max(), 4)

    # Active emotion: present if max score >= threshold in at least one post
    # This captures episodic emotions rather than averaging them away
    active = [e for e in ALL_EMOTIONS if scores.get(f"max_{e}", 0) >= THRESHOLD]

    return pd.Series({
        **scores,
        "active_emotions": "|".join(active) if active else "neutral"
    })

df_user_emotions = df_with_emotions.groupby("author_anon").apply(
    aggregate_emotions, include_groups=False
).reset_index()

print(f"User-level emotions: {len(df_user_emotions)} users")
print(f"\nActive emotion distribution (max >= {THRESHOLD}):")
all_active = []
for e in df_user_emotions["active_emotions"]:
    all_active.extend(e.split("|"))
print(pd.Series(collections.Counter(all_active)).sort_values(ascending=False).to_string())

In [ ]:
# Merge and save final dataset
df_final = df_pooled.merge(df_user_emotions, on="author_anon", how="inner")
print(f"Final dataset: {len(df_final)} users | {len(df_final.columns)} columns")

output_path = "/content/drive/MyDrive/Raw Data/reddit_estrangement_pooled_emotions.csv"
df_final.to_csv(output_path, index=False, encoding="utf-8-sig")
print(f"✓ Saved: {output_path}")